In [6]:
from sklearn.preprocessing import StandardScaler

features = ['ma_200', 'rsi']
X = data[features]
y = data['target']

# Chronological Split (80% Training, 20% Testing) to prevent look-ahead bias
split = int(len(X) * 0.8)
X_train_raw, X_test_raw = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Standardization for coefficient stability in Ridge Regression
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Train Size: {X_train.shape[0]} | Test Size: {X_test.shape[0]}")

Train Size: 14676 | Test Size: 3670


In [7]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Stage 1: Ridge Regression (L2-Regularized Linear Baseline)
model_ridge = Ridge(alpha=100.0)
model_ridge.fit(X_train, y_train)

# Stage 2: Random Forest for Residual Correction
# Modeling the errors (residuals) left by the linear model
train_residuals = y_train - model_ridge.predict(X_train)
model_rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
model_rf.fit(X_train, train_residuals)

# Final Synthesis: Baseline Prediction + Non-linear Correction
final_predictions = model_ridge.predict(X_test) + model_rf.predict(X_test)
print("Hybrid Architecture Training Phase Complete.")

Hybrid Architecture Training Phase Complete.


In [8]:
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Calculate Evaluation Metrics
r2 = r2_score(y_test, final_predictions)
rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
pcc, _ = pearsonr(y_test, final_predictions)
scc, _ = spearmanr(y_test, final_predictions)

# Output Results in a Structured Format
print("-" * 45)
print(f"{'Performance Metric':<25} | {'Value':<15}")
print("-" * 45)
print(f"{'R-squared (R2)':<25} | {r2:.4f}")
print(f"{'RMSE (SAR)':<25} | {rmse:.4f}")
print(f"{'Pearson Corr (PCC)':<25} | {pcc:.4f}")
print(f"{'Spearman Corr (SCC)':<25} | {scc:.4f}")
print("-" * 45)

---------------------------------------------
Performance Metric        | Value          
---------------------------------------------
R-squared (R2)            | 0.9345
RMSE (SAR)                | 79.2667
Pearson Corr (PCC)        | 0.9710
Spearman Corr (SCC)       | 0.9772
---------------------------------------------
